# Crossmatch_pipeline

- First calculate the angular distance between each X-ray source and all optical sources.
- Select optical sources within the 95% confidence radius (r95) of the X-ray source.
- If no sources are found within r95, apply a fallback search using a fixed radius of 2 arcseconds.
- For each matched candidate, compile a row of data including positional information, photometric measurements, and a simple classification based on Mv and log_Lx.
- If no candidates are found for an X-ray source, it is skipped.
- The final output is a DataFrame containing all matched candidates with their associated data.

In [44]:
import sys
import os
import numpy as np
import pandas as pd
sys.path.append(os.path.abspath(".."))

from src.optical_pipeline.classify_optical import optical_classification
from src.xray_pipeline.classify_xray import x_ray_classification
from src.load_cluster_data import load_cluster_config

## Defining important functions

### 1) compute_angular_distance
Calculate the angular distance of the optical source and the xray source (degrees).

In [45]:
def compute_angular_distance(ra, dec, ra0, dec0):
    
    return np.sqrt(
        ((ra - ra0) * np.cos(np.deg2rad(dec)))**2 +
        (dec - dec0)**2
    )

### 2) classify_optical_xray
Simple classification based on Mv and log_Lx, 34 and 36.2 are empirically derived thresholds. Those values can be found onthe next paper:
https://www.cambridge.org/core/journals/proceedings-of-the-international-astronomical-union/article/observational-evidence-for-the-origin-of-xray-sources-in-globular-clusters/23A31D04F0D308EC4B1CC2727290ECE8


In [46]:
def classify_optical_xray(Mv, log_Lx):

    z = 0.4 * Mv + log_Lx
     
    if z < 34:
        return "AB"
    elif z < 36.2:
        return "CV"
    else:
        return "LMXRB"

### 3) build_row
Build a row for the matched candidate, including classification and photometric data.

In [47]:
def build_row(CX, num, i_opt, optical_df, xray_df, radius_flag, n_sources):

    Mv = optical_df.loc[i_opt, "Mv"]
    log_Lx = xray_df.loc[CX, "log_LX_soft"]

    classification = classify_optical_xray(Mv, log_Lx)

    return {
        "id_candidate": f"{CX}_{num+1}",
        "Xray source": CX,
        "opt_id": optical_df.loc[i_opt, "Id"],
        "Hardness_classification": xray_df.loc[CX, "class"],
        "Mv vs xray": classification,

        # Photometry
        "pos_0": optical_df.loc[i_opt, "position_0"],
        "pos_1": optical_df.loc[i_opt, "position_1"],
        "pos_2": optical_df.loc[i_opt, "position_2"],

        # Matching info
        "radius": radius_flag,
        "n_sources": n_sources
    }

## Loading data

In [48]:
params = load_cluster_config("NGC_6809")

optical_df = optical_classification(distance_parsecs = params["distance_parsecs"], 
                      path_optical = params["path_optical"])

xray_df = x_ray_classification(
    path_xray=params["path_xray"],
    BS_RA = params["BS_RA"],
    BS_Decl = params["BS_Decl"],
    distance_parsecs=params["distance_parsecs"]
)
cx_list = xray_df.index.tolist()

Optical classification completed
X-ray classification completed


## run_crossmatch_pipeline

In [49]:
results = []

# Pre-extract arrays (faster)
ra = optical_df["RA"].values
dec = optical_df["Decl"].values

for CX in cx_list:

    ra0 = xray_df.loc[CX, "RA"]
    dec0 = xray_df.loc[CX, "Decl"]
    r95 = xray_df.loc[CX, "r95"]

    dist = compute_angular_distance(ra, dec, ra0, dec0)

    # --- FIRST SEARCH: Sources inside r95 degrees ---
    idx = np.where(dist < r95)[0]
    radius_flag = "r95"

    # --- FALLBACK: 2 arcsec ---
    radius = 2 / 3600  # Convert arcsec to degrees
    if len(idx) == 0 and r95 < radius:
        idx = np.where(dist < radius)[0]
        radius_flag = "2"

    # --- PROCESS MATCHES ---
    n_sources = len(idx)

    if n_sources == 0:
        pass

    for num, i_opt in enumerate(idx):

        row = build_row(
            CX,
            num,
            i_opt,
            optical_df,
            xray_df,
            radius_flag,
            n_sources
        )

        results.append(row)

df_matches =  pd.DataFrame(results)

## Final result

In [50]:
df_matches

,id_candidate,Xray source,opt_id,Hardness_classification,Mv vs xray,pos_0,pos_1,pos_2,radius,n_sources
0,CX2_1,CX2,R0044635,CV & AB,AB,MS,MS,MS,2,4
1,CX2_2,CX2,R0044636,CV & AB,AB,MS,MS,MS,2,4
2,CX2_3,CX2,R0044637,CV & AB,CV,bluer than MS L1,MS,MS,2,4
3,CX2_4,CX2,R0045252,CV & AB,CV,MS,MS,MS,2,4
4,CX8_1,CX8,R0000339,CV & AB,AB,bluer than SGB,RGB,RGB,2,2
5,CX8_2,CX8,R0006914,CV & AB,CV,below the MS,MS,MS,2,2
6,CX9_1,CX9,R0002365,CV & AB,AB,bluer than MSTO,bluer than MSTO,bluer than MSTO,2,6
7,CX9_2,CX9,R0002374,CV & AB,AB,MSTO,MSTO,Sub Giant Branch,2,6
8,CX9_3,CX9,R0031224,CV & AB,AB,MS,MS,MS,2,6
9,CX9_4,CX9,R0032091,CV & AB,AB,MS,MS,MS,2,6
